# **Upload Datasets**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from glob import glob
import json
from tqdm import tqdm
import random
import cv2
import imageio
import matplotlib.pyplot as plt
import os 
from sklearn.model_selection import train_test_split
import shutil
import random
import yaml
from distutils.dir_util import copy_tree

In [ ]:
!unzip "/content/drive/MyDrive/zip data/dermotology_datasets_organized_augmentation.zip"

# **Data Process**

In [ ]:
trainimages = glob(data_images_path +"/*.jpg")
trainlabels = glob(data_labels_path +"/*.txt")
print(len(trainimages))
print(len(trainlabels))


In [ ]:
os.mkdir("/content/datasets")
os.mkdir("/content/datasets/images")
os.mkdir("/content/datasets/labels")

os.mkdir("/content/datasets/images/train")
os.mkdir("/content/datasets/images/test")
os.mkdir("/content/datasets/images/valid")


os.mkdir("/content/datasets/labels/train")
os.mkdir("/content/datasets/labels/test")
os.mkdir("/content/datasets/labels/valid")


dataset_train_path = "/content/datasets/images/train"
dataset_valid_path ="/content/datasets/images/valid"
dataset_test_path = "/content/datasets/images/test"




In [ ]:
train_image, test_image= train_test_split(trainimages,shuffle=True, test_size=0.25, random_state=42)
print("train count",len(train_image))
print("test count",len(test_image))


In [ ]:
train_image, valid_image= train_test_split(train_image,shuffle=True, test_size=0.2, random_state=42)
print("train count",len(train_image))
print("valid count",len(valid_image))

In [ ]:
print(len(train_image))
print(len(test_image))
print(len(valid_image))

In [ ]:
for train in train_image:
  shutil.copy(train,dataset_train_path)

for valid in valid_image:
  shutil.copy(valid,dataset_valid_path)

for test in test_image:
  shutil.copy(test,dataset_test_path)


In [ ]:
datasetrain = glob(dataset_train_path +"/*.jpg")
datasetvalid = glob(dataset_valid_path + "/*.jpg")
datasettest = glob(dataset_test_path + "/*.jpg")
print(len(datasetrain))
print(len(datasetvalid))
print(len(datasettest))

In [ ]:
trainlabels[0]

In [ ]:
trainlabels[0].split(".txt")[0].split("/")[-1]

In [ ]:
datasetrain[0]

In [ ]:
datasetrain[0].split("/")[5].split(".jpg")[0]

In [ ]:
datasetrain[0].split(".jpg")[0].split("/")[-1]

In [ ]:
for labels in trainlabels:
  for trainlabels in datasetrain:
    if(labels.split(".txt")[0].split("/")[-1] == trainlabels.split(".jpg")[0].split("/")[-1]):
      shutil.copy(labels,"/content/datasets/labels/train")

trainlabels = glob(data_labels_path + "/*.txt")
for labels in trainlabels:
  for testlabels in datasettest:
    if(labels.split(".txt")[0].split("/")[-1] == testlabels.split(".jpg")[0].split("/")[-1]):
      shutil.copy(labels,"/content/datasets/labels/test")

trainlabels = glob(data_labels_path + "/*.txt")

for labels in trainlabels:
  for validlabels in datasetvalid:
    if(labels.split(".txt")[0].split("/")[-1] == validlabels.split(".jpg")[0].split("/")[-1]):
      shutil.copy(labels,"/content/datasets/labels/valid")

In [ ]:
#os.mkdir("/content/drive/MyDrive/yolov5_results")
os.mkdir("/content/drive/MyDrive/yolov5_results/"+dataname)


In [ ]:
dataname

In [ ]:
!zip -r bu_data_son.zip /datasets

In [ ]:
shutil.copy("/content/yolov5/bu_data_son.zip","/content/drive/MyDrive/zip files") 

# **Yolo v5**

In [ ]:
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
%pip install -qr requirements.txt  # install

import torch
import utils
display = utils.notebook_init()  # checks

In [ ]:
numberclass = 3
class_name = ["canis","rubrum","tonsurans"]
data = dict(
    nc = numberclass,
    names = class_name,
    path = '/content/datasets', #/content/datasets
    train = 'images/train',
    val = 'images/valid',
    test = 'images/test',
    )


with open('/content/yolov5/data/data.yaml', 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)



In [ ]:
pngimage = cv2.imread("/content/datasets/images/test/20210318_173003.jpg")
pngimage = cv2.cvtColor(pngimage,cv2.COLOR_BGR2RGB)
plt.figure(figsize = (8,8))
plt.axis("off")
shape = pngimage.shape
plt.title(shape)
plt.imshow(pngimage)

# **Training**

*(demo image — run notebook to regenerate)*

In [ ]:
dataname = "dermotology_datasets_organized_augmentation_128_4_100_yolov5n6"

In [ ]:
!python train.py --img 1280 --batch 4 --workers 2 --epochs 100 --data data.yaml --weights yolov5n6.pt --cache   

In [ ]:
copy_tree("/content/yolov5/runs", "/content/drive/MyDrive/yolov5_results/"+ dataname +"/runs") #/content/drive/MyDrive/runss     /content/yolov5/runs

# **TRAİN RESULTS**

In [ ]:
#copy_tree("/content/drive/MyDrive/yolov5_results/02.02.2023--dermatology_dataset_640_4_100_yolov5x", "/content/yolov5")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/train

In [ ]:
import pandas as pd

df = pd.read_csv('/content/yolov5/runs/train/exp/results.csv')
df = df.tail(1)
df = df.drop(["               epoch","      train/obj_loss","      train/cls_loss","        val/box_loss",
              "               x/lr0","               x/lr1","               x/lr2","      train/box_loss",
              "        val/obj_loss","        val/cls_loss"],axis=1)
df

In [ ]:
def res_vis(exp_path,png):

  pnglist = glob(exp_path+"/*."+png) 

  for img in pnglist:
    imagename = img.split("/")[6].split(".png")[0]

    pngimage = cv2.imread(img)
    pngimage = cv2.cvtColor(pngimage,cv2.COLOR_BGR2RGB)
    plt.figure(figsize = (8,8))
    plt.axis("off")
    plt.title(imagename)
    plt.imshow(pngimage)

In [ ]:
res_vis("/content/yolov5/runs/train/exp","png")

In [ ]:
res_vis("/content/yolov5/runs/train/exp","jpg")

In [ ]:
result = cv2.imread("/content/yolov5/runs/train/exp/results.png")
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.figure(figsize = (20,20))
plt.axis("off")
plt.title("result")
plt.imshow(result)


# **Testing**

In [ ]:
bestweights = "/content/yolov5/runs/train/exp/weights/best.pt"  #"/content/yolov5/runs/train/exp/weights/best.pt"
lastweights = "/content/yolov5/runs/train/exp/weights/last.pt"
shutil.copy(bestweights,"/content/yolov5")
shutil.copy(lastweights,"/content/yolov5")

In [ ]:
!python val.py --weights best.pt --data data.yaml --img 1280 --iou 0.65    --save-json --save-conf  --half --task test --save-txt

# **TEST RESULTS**

In [ ]:
copy_tree("/content/yolov5/runs", "/content/drive/MyDrive/yolov5_results/"+ dataname +"/runs")

In [ ]:
res_vis("/content/yolov5/runs/val/exp","png")

In [ ]:
res_vis("/content/yolov5/runs/val/exp","jpg")

# **TEST DETECT**

In [ ]:
!python detect.py --weights best.pt --img 640 --conf 0.7 --iou-thres 0.7 --save-txt --source /content/datasets/images/test 


In [ ]:
pnglist = glob("/content/yolov5/runs/detect/exp/*.jpg") 
for img in random.sample(pnglist,20):
  imagename = img.split("/")[-1].split(".png")[0]
  pngimage = cv2.imread(img)
  pngimage = cv2.cvtColor(pngimage,cv2.COLOR_BGR2RGB)
  plt.figure(figsize = (8,8))
  plt.axis("off")
  plt.title(imagename)
  plt.imshow(pngimage)

In [ ]:
copy_tree("/content/yolov5/runs", "/content/drive/MyDrive/yolov5_results/"+ dataname +"/runs")

In [ ]:
def yolo_to_pascal(bbox, w, h):
   
    w_half_len = (bbox[2] * w) / 2
    h_half_len = (bbox[3] * h) / 2
    xmin = int((bbox[0] * w) - w_half_len)
    ymin = int((bbox[1] * h) - h_half_len)
    xmax = int((bbox[0] * w) + w_half_len)
    ymax = int((bbox[1] * h) + h_half_len)
    return [xmin, ymin, xmax, ymax]

In [ ]:
def vis_iou(size,iouthreshold,jpeg_list_path,png_or_jpg,text_path,predict_txt_path,num_of_sample):
  size = size
  iouthreshold = iouthreshold

  jpeg_list = glob( jpeg_list_path +"/*." + png_or_jpg)  # glob("/content/datasets/images/test/*.jpg")


  for i in  random.sample(jpeg_list,num_of_sample):  #random.sample(jpeg_list,20)
    img = imageio.imread(i)
    img = cv2.resize(img,(size,size))
    id_jpeg = i.split("/")[5].split("."+png_or_jpg)[0]


    txt_file = text_path + "/" + id_jpeg +".txt" #"/content/datasets/labels/test/" + id_jpeg +".txt"

    with open(txt_file,'r') as f:
      img_anns = f.readlines()
    try:
      for index in img_anns:
        x = float(index.split(" ")[1])
        y = float(index.split(" ")[2])
        h = float(index.split(" ")[3])
        w = float(index.split(" ")[4])
        bbox = [x,y,h,w]

        x1,y1,x2,y2 = yolo_to_pascal(bbox,size,size)
        cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0), 2)

        predict_txt = predict_txt_path +"/" + id_jpeg + ".txt" #"/content/yolov5/runs/detect/exp/labels/" + id_jpeg + ".txt"

        with open(predict_txt,'r') as f:
            predictlines = f.readlines()
        for index in predictlines:
          # print(i.split(" "))
          classnumber = index.split(" ")[0]
          xp = float(index.split(" ")[1])
          yp = float(index.split(" ")[2])
          hp = float(index.split(" ")[3])
          wp = float(index.split(" ")[4])
          bboxp = [xp,yp,hp,wp]
          xp1,yp1,xp2,yp2 = yolo_to_pascal(bboxp,size,size)
          xA = max(x1, xp1)
          yA = max(y1, yp1)
          xB = min(x2, xp2)
          yB = min(y2, yp2)
          interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
          boxAArea = (x2 - x1 + 1) * (y2 - y1 + 1)
          boxBArea = (xp2 - xp1 + 1) * (yp2 - yp1 + 1)
          iou = interArea / float(boxAArea + boxBArea - interArea)
          if(iou>iouthreshold):
            cv2.rectangle(img,(xp1,yp1),(xp2,yp2),(255,0,0), 2)
            cv2.putText(img, " iou = {:.2f}".format(iou), (xp1,yp1-10), cv2.FONT_HERSHEY_SIMPLEX,0.5, (255,0,0),1, cv2.LINE_AA) 

    except FileNotFoundError as e:
      print("")   

    plt.figure(figsize = (9,9))
    plt.title(id_jpeg)   
    plt.axis("off")
    plt.imshow(img) 

In [ ]:
vis_iou(640,0.5,"/content/datasets/images/test" ,"jpg"
,"/content/datasets/labels/test","/content/yolov5/runs/detect/exp/labels",20)

In [ ]:
size = 1024
i = "/content/content/datasets/images/test/20210318_173003_45.jpg"
img = imageio.imread(i)
img = cv2.resize(img,(size,size))
id_jpeg = i.split("/")[6].split(".jpg")[0]

   
txt_file = "/content/content/datasets/labels/test" + id_jpeg +".txt"
with open(txt_file,'r') as f:
  img_anns = f.readlines()
  
try:
  for index in img_anns:
    
    x = float(index.split(" ")[1])
    y = float(index.split(" ")[2])
    h = float(index.split(" ")[3])
    w = float(index.split(" ")[4])
    
    bbox = [x,y,h,w]
    x1,y1,x2,y2 = yolo_to_pascal(bbox,size,size)
    cv2.rectangle(img,(x1,y1),(x2,y2),(0,0,255), 2)
    predict_txt = "/content/drive/MyDrive/mali/new_classes/detect/exp/labels/" + id_jpeg + ".txt"
    with open(predict_txt,'r') as f:
        predictlines = f.readlines()
        print(predictlines)
    for index in predictlines:
     
      classnumber = index.split(" ")[0]
      xp = float(index.split(" ")[1])
      yp = float(index.split(" ")[2])
      hp = float(index.split(" ")[3])
      wp = float(index.split(" ")[4])
      bboxp = [xp,yp,hp,wp]
      xp1,yp1,xp2,yp2 = yolo_to_pascal(bboxp,size,size)
      xA = max(x1, xp1)
      yA = max(y1, yp1)
      xB = min(x2, xp2)
      yB = min(y2, yp2)
      interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
      boxAArea = (x2 - x1 + 1) * (y2 - y1 + 1)
      boxBArea = (xp2 - xp1 + 1) * (yp2 - yp1 + 1)
      iou = interArea / float(boxAArea + boxBArea - interArea)
      if(iou>0.5):
        cv2.rectangle(img,(xp1,yp1),(xp2,yp2),(255,0,0), 2)
        cv2.putText(img, " iou = {:.2f}".format(iou), (xp1,yp1-10), cv2.FONT_HERSHEY_SIMPLEX,0.5, (255,0,0),1, cv2.LINE_AA) 
        
except FileNotFoundError as e:
  print("")   
plt.figure(figsize = (8,8))
plt.title(id_jpeg)   
plt.axis("off")
plt.imshow(img)

# **Appendix**

*(demo image — run notebook to regenerate)*

In [ ]:
# https://github.com/ultralytics/yolov5
for b, m in zip([64, 64, 64, 32, 16], ['yolov5n', 'yolov5s', 'yolov5m', 'yolov5l', 'yolov5x']):  # batch, model
  !python train.py --batch {b} --weights {m}.pt --data VOC.yaml --epochs 50 --img 512 --hyp hyp.VOC.yaml --project TRAINING --name {m} --cache

In [ ]:
# Reproduce
for x in 'yolov5n', 'yolov5s', 'yolov5m', 'yolov5l', 'yolov5x':
  !python val.py --weights {x}.pt --data coco.yaml --img 640 --task speed  # speed
  !python val.py --weights {x}.pt --data coco.yaml --img 640 --conf 0.001 --iou 0.65  # mAP

# SAHI 

In [ ]:
!pip install -U torch sahi yolov5

In [ ]:
import os
os.getcwd()

In [ ]:
from sahi.utils.cv import read_image
from sahi.utils.file import download_from_url
from sahi.predict import get_prediction, get_sliced_prediction, predict
from IPython.display import Image
from sahi.model import DetectionModel
from sahi.prediction import ObjectPrediction
from typing import Dict, List, Optional, Union
from sahi.utils.compatibility import fix_full_shape_list, fix_shift_amount_list

In [ ]:
CUSTOM_YOLO5_CLASS = True

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sys
import cv2
import torch
from PIL import Image as Img
from IPython.display import display

In [ ]:
from IPython.core.magic import (register_line_cell_magic)

In [ ]:
# not neccessary 
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__) 

In [ ]:
class YourYolov5DetectionModel(DetectionModel):

  def load_model(self):

      try:
          import torch
      except ImportError:
          raise ImportError(
              "Please install torch and Your-YOLOv5 first for YOLOv5 inference."
          )

      # set model
      try:
          model = torch.hub.load("/content/yolov5", 
                        'custom', 
                        path=self.model_path,
                        source='local',
                        force_reload=True)
          model.conf = self.confidence_threshold
          self.model = model
      except Exception as e:
          TypeError("model_path is not a valid yolov5 model path: ", e)

      # set category_mapping
      if not self.category_mapping:
          category_mapping = {
              str(ind): category_name
              for ind, category_name in enumerate(self.category_names)
          }
          self.category_mapping = category_mapping

  def perform_inference(self, image: np.ndarray, image_size: int = None):
      """
      Prediction is performed using self.model and the prediction result is set to self._original_predictions.
      Args:
          image: np.ndarray
              A numpy array that contains the image to be predicted. 3 channel image should be in RGB order.
          image_size: int
              Inference input size.
      """
      try:
          import yolov5
      except ImportError:
          raise ImportError('Please run "pip install -U yolov5" ' "to install YOLOv5 first for YOLOv5 inference.")

      # Confirm model is loaded
      assert self.model is not None, "Model is not loaded, load it by calling .load_model()"

      if image_size is not None:
          warnings.warn("Set 'image_size' at DetectionModel init.", DeprecationWarning)
          prediction_result = self.model(image, size=image_size, augment=True)
      elif self.image_size is not None:
          prediction_result = self.model(image, size=self.image_size)
      else:
          prediction_result = self.model(image)

      self._original_predictions = prediction_result

  @property
  def num_categories(self):
      """
      Returns number of categories
      """
      return len(self.model.names)

  @property
  def has_mask(self):
      """
      Returns if model output contains segmentation mask
      """
      has_mask = self.model.with_mask
      return has_mask

  @property
  def category_names(self):
      return self.model.names

  def _create_object_prediction_list_from_original_predictions(
      self,
      shift_amount_list: Optional[List[List[int]]] = [[0, 0]],
      full_shape_list: Optional[List[List[int]]] = None,
  ):
      """
      self._original_predictions is converted to a list of prediction.ObjectPrediction and set to
      self._object_prediction_list_per_image.
      Args:
          shift_amount_list: list of list
              To shift the box and mask predictions from sliced image to full sized image, should
              be in the form of List[[shift_x, shift_y],[shift_x, shift_y],...]
          full_shape_list: list of list
              Size of the full image after shifting, should be in the form of
              List[[height, width],[height, width],...]
      """
      original_predictions = self._original_predictions

      # compatilibty for sahi v0.8.15
      shift_amount_list = fix_shift_amount_list(shift_amount_list)
      full_shape_list = fix_full_shape_list(full_shape_list)

      # handle all predictions
      object_prediction_list_per_image = []
      for image_ind, image_predictions_in_xyxy_format in enumerate(original_predictions.xyxy):
          shift_amount = shift_amount_list[image_ind]
          full_shape = None if full_shape_list is None else full_shape_list[image_ind]
          object_prediction_list = []

          # process predictions
          for prediction in image_predictions_in_xyxy_format.cpu().detach().numpy():
              x1 = int(prediction[0])
              y1 = int(prediction[1])
              x2 = int(prediction[2])
              y2 = int(prediction[3])
              bbox = [x1, y1, x2, y2]
              score = prediction[4]
              category_id = int(prediction[5])
              
              if category_id == 0:
                category_name = "canis"
                                                      #['canis','rubrum','tonsurans']
              if category_id == 1:
                category_name = "rubrum"

              if category_id == 2:
                category_name = "tonsurans"


              # ignore invalid predictions
              if bbox[0] > bbox[2] or bbox[1] > bbox[3] or bbox[0] < 0 or bbox[1] < 0 or bbox[2] < 0 or bbox[3] < 0:
                  logger.warning(f"ignoring invalid prediction with bbox: {bbox}")
                  continue
              if full_shape is not None and (
                  bbox[1] > full_shape[0]
                  or bbox[3] > full_shape[0]
                  or bbox[0] > full_shape[1]
                  or bbox[2] > full_shape[1]
              ):
                  logger.warning(f"ignoring invalid prediction with bbox: {bbox}")
                  continue

              object_prediction = ObjectPrediction(
                  bbox=bbox,
                  category_id=category_id,
                  score=score,
                  bool_mask=None,
                  category_name=category_name,
                  shift_amount=shift_amount,
                  full_shape=full_shape,
              )
              object_prediction_list.append(object_prediction)
          object_prediction_list_per_image.append(object_prediction_list)

      self._object_prediction_list_per_image = object_prediction_list_per_image 

In [ ]:
detection_model = YourYolov5DetectionModel(
    
    model_path= "/content/yolov5/best.pt",
    confidence_threshold=0.5,
    device="cuda:0", # or 'cuda:0'
)

In [ ]:
for path in random.sample((glob("/content/datasets/labels/test/*")),10):    
    
    result = get_sliced_prediction(
        path,
        detection_model,
        slice_height = 512,
        slice_width = 512,
        overlap_height_ratio = 0.2,
        overlap_width_ratio = 0.2
    )

    result.export_visuals(export_dir="demo_data/")

    Image("demo_data/prediction_visual.png")